# ESM3-OPEN FUNC PRED

## Install Dependencies (Optional)

This section ensures that all necessary packages are installed in your Conda environment. We use `conda` for package management and install additional packages using `pip` when needed.

If you are following this guide from scratch, uncomment the provided installation commands to set up your environment. For reproducibility, we also include instructions to list the installed packages in your Conda environment.

In [1]:
# Install necessary packages using Conda
# Uncomment the following lines to install them if not already done:
# !conda install -c conda-forge python=3.12 pip -y
# !conda install -c conda-forge biopython -y
# !conda install -c conda-forge py3dmol -y

# Install additional packages using pip
# Uncomment the following lines if needed:
# !pip install huggingface_hub
# !pip install git+https://github.com/facebookresearch/esm.git

# Display installed packages in the Conda environment
!conda list

# packages in environment at /Users/admin/opt/anaconda3/envs/esm:
#
# Name                    Version                   Build  Channel
anyio                     4.8.0              pyhd8ed1ab_0    conda-forge
appnope                   0.1.4              pyhd8ed1ab_1    conda-forge
argon2-cffi               23.1.0             pyhd8ed1ab_1    conda-forge
argon2-cffi-bindings      21.2.0          py312hb553811_5    conda-forge
arrow                     1.3.0              pyhd8ed1ab_1    conda-forge
asttokens                 3.0.0              pyhd8ed1ab_1    conda-forge
async-lru                 2.0.4              pyhd8ed1ab_1    conda-forge
attrs                     24.3.0             pyh71513ae_0    conda-forge
babel                     2.16.0             pyhd8ed1ab_1    conda-forge
beautifulsoup4            4.12.3             pyha770c72_1    conda-forge
biopython                 1.84            py312hbd25219_0    conda-forge
biotite                   0.41.2                   pypi_0    p

## Imports

This section includes all the necessary Python libraries for the workflow. Each import is grouped based on its purpose, such as handling sequences, structures, models, or visualization.

In [1]:
# Core Utilities
import os  # For environment settings and file handling
import torch
torch.set_default_dtype(torch.float32)

# Bioinformatics Libraries
from Bio.PDB import PDBParser, PDBList  # For parsing PDB files and extracting sequences

# ESM Model and Hugging Face Utilities
import huggingface_hub  # To authenticate with Hugging Face and download models
from esm.models.esm3 import ESM3  # For loading and working with the esm3-open model
from esm.sdk.api import ESM3InferenceClient, ESMProtein, GenerationConfig  # For handling protein sequences and model generation tasks
from esm.utils.structure.protein_chain import ProteinChain

# Visualization
import py3Dmol  # For visualizing 3D protein structures

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


## Download Model and Tokenizers (Optional)

In this section, we download the model.

In [3]:
# Step 1: Authenticate with Hugging Face
# Uncomment the following line to enter your token directly
YOUR_HUGGING_FACE_TOKEN = "HF_TOKEN_REDACTED"
huggingface_hub.login(token=YOUR_HUGGING_FACE_TOKEN, add_to_git_credential=False)

# For interactive login (recommended if no token is stored)
# login()

# Step 2: Initialize the esm3-open model
model: ESM3InferenceClient = ESM3.from_pretrained("esm3-open").to("mps")  # Use 'cuda' or 'cpu' if applicable
print("ESM3-open model initialized successfully!")

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

ESM3-open model initialized successfully!


## Offline Model Initialization

This section ensures the `esm3-open` model is loaded offline. We verify its existence in the local Hugging Face cache and initialize it for predictions. The environment is configured to prevent online access during execution.

In [2]:
try:
    # Attempt to load the model to determine if it exists locally
    model = False
    model = ESM3.from_pretrained("esm3-open").to("cpu")
    if model != False:  # Extract the parent directory where the model is cached
        print(f"Model is available locally!")
except FileNotFoundError:
    # If the model is not cached locally, prompt for download
    print("The esm3-open model is not found locally.")
    print("If you want to download the model, run the following:")
    print("""
    from huggingface_hub import login
    login()
    from esm.models.esm3 import ESM3
    model = ESM3.from_pretrained("esm3-open")
    """)
    raise

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/3.06k [00:00<?, ?B/s]

Model is available locally!


## Retrieve Protein Structure and Sequence

In this section, we use the `ProteinChain` class from the ESM SDK to fetch a specific protein structure and sequence from the PDB. This ensures the sequence and structure information are directly sourced without preprocessing. 

We visualize the protein using `py3Dmol` to confirm it has been loaded correctly.

In [3]:
# Define input mode: "fetch" (from PDB database) or "local" (local PDB file)
input_mode = "fetch"  # Options: "fetch" or "local"

# Initialize variables
protein_chain = None

if input_mode == "fetch":
    # Fetch structure and sequence from the PDB database
    pdb_id = "6ZBS"  # Replace with your desired PDB ID
    chain_id = "A"   # Chain ID to extract
    print(f"Fetching PDB ID: {pdb_id}, Chain: {chain_id}")

    protein_chain = ProteinChain.from_rcsb(pdb_id, chain_id)

elif input_mode == "local":
    # Load structure and sequence from a local PDB file
    pdb_filename = "3_HIBD.pdb"  # Replace with your desired PDB filename
    pdb_folder = "PDB_files"
    pdb_path = os.path.join(pdb_folder, pdb_filename)

    if not os.path.exists(pdb_path):
        raise FileNotFoundError(f"PDB file not found at: {pdb_path}")

    protein_chain = ProteinChain.from_pdb(pdb_path, chain_id="A")

else:
    raise ValueError(f"Invalid input mode: {input_mode}. Choose 'fetch' or 'local'.")

# Visualize the structure using Py3Dmol
view = py3Dmol.view(width=300, height=200)
pdb_str = protein_chain.to_pdb_string()  # Convert the ProteinChain object to a PDB string
view.addModel(pdb_str, "pdb")  # Load the PDB string into the Py3Dmol viewer
view.setStyle({"cartoon": {"color": "spectrum"}})  # Set the style to cartoon with spectrum coloring
view.zoomTo()  # Zoom in on the protein chain
view.show()  # Display the protein chain

Fetching PDB ID: 6ZBS, Chain: A


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
# Initialize ESMProtein Object
protein = ESMProtein.from_protein_chain(protein_chain)
print("Protein loaded successfully.")

# Clear any existing annotations
protein.function_annotations = None  # Clear function annotations if present

# Configure Function Prediction
sequence_length = len(protein.sequence)  # Length of the protein sequence
num_steps = max(1, int(sequence_length / 64))  # Dynamically determine num_steps
print(f"Function prediction will use {num_steps} steps based on sequence length.")

generation_config = GenerationConfig(
    track="function", schedule="cosine", strategy="entropy", num_steps=num_steps
)

# Generate Function Prediction
protein_with_function = model.generate(protein, generation_config)

Protein loaded successfully.
Function prediction will use 6 steps based on sequence length.


100%|██████████| 6/6 [02:45<00:00, 27.61s/it]


In [5]:
# Display Prediction Results
print("\nPredicted Function Annotations:")
if protein_with_function.function_annotations:
    for annotation in protein_with_function.function_annotations:
        print(f"Function: {annotation.label}, Start: {annotation.start}, End: {annotation.end}")
else:
    print("No function annotations predicted.")


Predicted Function Annotations:
Function: a protein, Start: 9, End: 194
Function: a protein, Start: 196, End: 196
Function: a protein, Start: 199, End: 199
Function: a protein, Start: 212, End: 237
Function: a protein, Start: 239, End: 414
Function: a protein, Start: 416, End: 416
Function: acetyltransferase, Start: 9, End: 194
Function: acetyltransferase, Start: 196, End: 196
Function: acetyltransferase, Start: 199, End: 199
Function: acetyltransferase, Start: 212, End: 414
Function: acetyltransferase, Start: 416, End: 416
Function: acetyltransferase like, Start: 9, End: 194
Function: acetyltransferase like, Start: 196, End: 196
Function: acetyltransferase like, Start: 199, End: 199
Function: acetyltransferase like, Start: 212, End: 414
Function: acetyltransferase like, Start: 416, End: 416
Function: atp binding, Start: 9, End: 194
Function: atp binding, Start: 196, End: 196
Function: atp binding, Start: 199, End: 199
Function: atp binding, Start: 212, End: 414
Function: atp binding,